In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

from huggingface_hub import list_repo_files

files = list_repo_files("yandex/yambda", repo_type="dataset")
print([f for f in files if "50m" in f])

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download("yandex/yambda", "flat/50m/listens.parquet", repo_type="dataset")
listens = pd.read_parquet(path)
print(listens.shape)
print(listens.head())

In [ ]:
print("Пользователей:", listens["uid"].nunique())
print("Треков:", listens["item_id"].nunique())
print("Событий:", len(listens))

In [ ]:
print("Min timestamp:", listens["timestamp"].min())
print("Max timestamp:", listens["timestamp"].max())

In [ ]:
import matplotlib.pyplot as plt

# Сколько событий прослушивания у каждого пользователя
user_counts = listens["uid"].value_counts()

# Сколько событий приходится на каждый трек
track_counts = listens["item_id"].value_counts()

print(user_counts.describe())
print(track_counts.describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

user_counts.hist(bins=100, ax=axes[0])
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Прослушивания на пользователя")
axes[0].set_xlabel("Количество прослушиваний")
axes[0].set_ylabel("Количество пользователей")

track_counts.hist(bins=100, ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Прослушивания на трек")
axes[1].set_xlabel("Количество прослушиваний")
axes[1].set_ylabel("Количество треков")



plt.tight_layout()
plt.show()



In [ ]:
organic_share = (listens["is_organic"] == 1).mean()
print(f"Доля органических прослушиваний: {organic_share:.2%}")

In [ ]:
r = listens["played_ratio_pct"]

print("Прослушивание < 10%:", (r < 10).mean().round(4))
print("Прослушивание < 20%:", (r < 20).mean().round(4))
print("Полные прослушивания:", (r == 100).mean().round(4))
print("Пропуски:", r.isna().mean().round(4))

In [ ]:
DAY = 86400          
GAP = 30 * 60

t_max = listens["timestamp"].max()
test_start = t_max - DAY
train_end  = test_start - GAP

# сигнал "понравилось": прослушано не менее 50% трека
a = listens["played_ratio_pct"] >= 50
pos = listens[a]

train = pos[pos["timestamp"] < train_end]
test  = pos[pos["timestamp"] >= test_start]

print(train.shape)
print(test.shape)




Метрики Recall@k, NDCG@k

In [ ]:
import numpy as np

def recall_at_k(recs, truth, k=100):
    # recs: dict uid -> список item_id по убыванию score
    # truth: dict uid -> set of item_id из теста
    scores = []
    for uid, t in truth.items():
        hits = len(set(recs.get(uid, [])[:k]) & t)
        scores.append(hits / len(t))
    return np.mean(scores)

def ndcg_at_k(recs, truth, k=100):
    scores = []
    for uid, t in truth.items():
        r = recs.get(uid, [])[:k]
        dcg = sum(1 / np.log2(i + 2) for i, it in enumerate(r) if it in t)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(t), k)))
        scores.append(dcg / idcg)
    return np.mean(scores)

Первый baseline: Most Popular

In [ ]:
truth = test.groupby("uid")["item_id"].apply(set).to_dict()
#Словарь вида uid -> {мн-во item_id, прослушанных треков}, из test
train_users = set(train["uid"])
#Доп проверка
truth = {u: s for u, s in truth.items() if u in train_users}

#top-100
top = train["item_id"].value_counts().index[:100].tolist()
#каждому пользователю рекомендуются просто top-100 треков oat
recs = {u: top for u in truth}

print("Recall@100:", recall_at_k(recs, truth))
print("NDCG@100:", ndcg_at_k(recs, truth))

In [ ]:
# print("train_end:", train_end, "test_start:", test_start, "t_max:", t_max)
# print("train:", train.shape, "test:", test.shape)
# print("test users before filter:", test["uid"].nunique())
# print("test users after filter:", len(truth))

In [ ]:
def split_at(pos, cutoff):
    train = pos[pos["timestamp"] < cutoff - GAP]
    target = pos[(pos["timestamp"] >= cutoff) & (pos["timestamp"] < cutoff + DAY)]
    truth = target.groupby("uid")["item_id"].apply(set).to_dict()
    users = set(train["uid"])
    truth = {u: s for u, s in truth.items() if u in users}
    return train, truth

def eval_popular(pos, cutoff, days=None, k=100):
    train, truth = split_at(pos, cutoff)
    if days is not None:
        train = train[train["timestamp"] >= cutoff - GAP - days * DAY]
    top = train["item_id"].value_counts().index[:k].tolist()
    recs = {u: top for u in truth}
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)

val_cutoff = test_start - DAY   # validation: сутки перед тестом

# for days in [None, 1, 3, 7, 14, 30]:
#     r, n = eval_popular(pos, val_cutoff, days)
#     print(f"window={days}: recall={r:.4f}, ndcg={n:.4f}")

In [ ]:
for days in [0.25, 0.5, 1, 3]:
    rs = []
    for shift in [1, 2, 3]:
        r, n = eval_popular(pos, test_start - shift * DAY, days)
        rs.append((r, n))
    print(days, np.mean(rs, axis=0))

# best_days = 1  
# print("TEST:", eval_popular(pos, test_start, best_days))

In [ ]:
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

#строит матрицу пользователи х треки, значение - кол-во прослушиваний
def build_matrix(train):
    uids = train["uid"].astype("category")
    iids = train["item_id"].astype("category")
    counts = train.groupby(["uid", "item_id"]).size().reset_index(name="n")
    u = counts["uid"].map({v: i for i, v in enumerate(uids.cat.categories)})
    i = counts["item_id"].map({v: j for j, v in enumerate(iids.cat.categories)})
    mat = sp.csr_matrix((counts["n"].astype("float32"), (u, i)),
                        shape=(len(uids.cat.categories), len(iids.cat.categories)))
    return mat, list(uids.cat.categories), list(iids.cat.categories)

#обучает и оценивает модель
def eval_als(pos, cutoff, factors=64, reg=0.01, iters=15, alpha=1.0, k=100):
    train, truth = split_at(pos, cutoff)
    seen = train.groupby("uid")["item_id"].apply(set).to_dict()
    truth_new = {u: t - seen[u] for u, t in truth.items()}
    truth_new = {u: t for u, t in truth_new.items() if t}

    mat, user_ids, item_ids = build_matrix(train)
    mat.data = 1 + alpha * mat.data
    model = AlternatingLeastSquares(factors=factors, regularization=reg,
                                iterations=iters, random_state=42)
    model.fit(mat)

    u_index = {u: i for i, u in enumerate(user_ids)}
    users = [u for u in truth_new if u in u_index]
    idx = np.array([u_index[u] for u in users])
    rec_idx, _ = model.recommend(idx, mat[idx], N=k, filter_already_liked_items=True)
    recs = {u: [item_ids[j] for j in row] for u, row in zip(users, rec_idx)}
    truth_new = {u: truth_new[u] for u in users}
    return recall_at_k(recs, truth_new, k), ndcg_at_k(recs, truth_new, k)

# print(eval_als(pos, test_start - DAY))

#Важно! filter_already_liked_items=True значит что нельзя рекомендовать уже услышанные треки, False - можно

In [ ]:
import itertools, pandas as pd
#Перебор вариантов гиперпараметров(grid search)(ДОЛГО)
# grid = {
#     "factors": [32, 64, 128],
#     "alpha":   [1, 10, 40],
#     "reg":     [0.01, 0.1],
# }

# rows = []
# for factors, alpha, reg in itertools.product(grid["factors"], grid["alpha"], grid["reg"]):
#     rs = [eval_als(pos, test_start - s * DAY, factors=factors, reg=reg, alpha=alpha)
#           for s in [1, 2, 3]]
#     r, n = np.mean(rs, axis=0)
#     rows.append(dict(factors=factors, alpha=alpha, reg=reg, recall=r, ndcg=n))
#     print(rows[-1])          # печатаем по ходу, чтобы видеть прогресс

# res = pd.DataFrame(rows).sort_values("ndcg", ascending=False)
# res.to_csv("als_grid.csv", index=False)   # сохрани, чтобы не потерять при падении ядра
# print(res.head(10))

In [ ]:
def eval_my_top(pos, cutoff, k=100):
    train, truth = split_at(pos, cutoff)
    top = (train.groupby(["uid", "item_id"]).size().reset_index(name="n")
                .sort_values(["uid", "n"], ascending=[True, False]))
    recs = top.groupby("uid")["item_id"].apply(lambda s: s.tolist()[:k]).to_dict()
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)

print(eval_my_top(pos, test_start - DAY))

In [ ]:
#Популярное без повторов
def eval_popular_new(pos, cutoff, days=1, k=100):
    train, truth = split_at(pos, cutoff)
    seen = train.groupby("uid")["item_id"].apply(set).to_dict()
    truth_new = {u: t - seen[u] for u, t in truth.items()}
    truth_new = {u: t for u, t in truth_new.items() if t}
    recent = train[train["timestamp"] >= cutoff - GAP - days * DAY]
    top = recent["item_id"].value_counts().index[:1000].tolist()
    recs = {u: [i for i in top if i not in seen[u]][:k] for u in truth_new}
    return recall_at_k(recs, truth_new, k), ndcg_at_k(recs, truth_new, k)

rs = [eval_popular_new(pos, test_start - s * DAY) for s in [1, 2, 3]]
print(rs)                      # по каждой отсечке отдельно
print(np.mean(rs, axis=0))     # [Recall, NDCG], усреднение по трём



# print(eval_popular_new(pos, val_cutoff))

In [ ]:
# train, truth = split_at(pos, val_cutoff)
# seen = train.groupby("uid")["item_id"].apply(set).to_dict()

# n_all = sum(len(t) for t in truth.values())
# n_new = sum(len(t - seen[u]) for u, t in truth.items())
# print("доля повторов среди (user, track) в тесте:", 1 - n_new / n_all)

# truth_new = {u: t - seen[u] for u, t in truth.items()}
# truth_new = {u: t for u, t in truth_new.items() if t}
# print("пользователей с новыми треками:", len(truth_new), "из", len(truth))
pos["timestamp"] = pos["timestamp"].astype("int64")

In [ ]:
def eval_my_top_decay(pos, cutoff, tau_days, k=100):
    train, truth = split_at(pos, cutoff)
    t_ref = float(cutoff - GAP)
    dt = t_ref - train["timestamp"].values.astype("float64")
    w = np.exp(-dt / (tau_days * DAY))
    assert w.max() <= 1.0 and np.isfinite(w).all()   # веса должны быть в (0, 1]
    s = (train[["uid", "item_id"]].assign(w=w)
              .groupby(["uid", "item_id"], as_index=False)["w"].sum()
              .sort_values(["uid", "w"], ascending=[True, False]))
    recs = s.groupby("uid")["item_id"].apply(lambda x: x.tolist()[:k]).to_dict()
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)



# for tau in [0.5, 1, 3, 7, 14, 30, 100]:
#     rs = [eval_my_top_decay(pos, test_start - s * DAY, tau) for s in [1, 2, 3]]
#     print(tau, np.mean(rs, axis=0))

In [ ]:
# rs = [eval_my_top(pos, test_start - s * DAY) for s in [1, 2, 3]]
# print(np.mean(rs, axis=0))

In [ ]:
#Расширение сетки

# grid2 = [(256, 1), (256, 0.3), (128, 0.3), (128, 0.1), (512, 0.3)]
# rows2 = []
# for factors, alpha in grid2:
#     rs = [eval_als(pos, test_start - s * DAY, factors=factors, reg=0.01, alpha=alpha)
#           for s in [1, 2, 3]]
#     r, n = np.mean(rs, axis=0)
#     rows2.append(dict(factors=factors, alpha=alpha, recall=r, ndcg=n))
#     print(rows2[-1])
#     pd.DataFrame(rows2).to_csv("als_grid2_partial.csv", index=False)

In [ ]:
def build_candidates(pos, cutoff, k_hist=200, k_als=100, k_pop=100,
                     tau=14, factors=256, alpha=0.3):
    train, truth = split_at(pos, cutoff)
    users = list(truth)

    # 1. история с затуханием
    dt = float(cutoff - GAP) - train["timestamp"].values.astype("float64")
    w = np.exp(-dt / (tau * DAY))
    s = (train[["uid", "item_id"]].assign(w=w)
              .groupby(["uid", "item_id"], as_index=False)["w"].sum()
              .sort_values(["uid", "w"], ascending=[True, False]))
    hist = s.groupby("uid")["item_id"].apply(lambda x: x.tolist()[:k_hist]).to_dict()

    # 2. iALS, только новые для пользователя треки
    mat, user_ids, item_ids = build_matrix(train)
    mat.data = 1 + alpha * mat.data
    model = AlternatingLeastSquares(factors=factors, regularization=0.01,
                                    iterations=15, random_state=42)
    model.fit(mat)
    u_index = {u: i for i, u in enumerate(user_ids)}
    idx = np.array([u_index[u] for u in users])
    rec_idx, _ = model.recommend(idx, mat[idx], N=k_als, filter_already_liked_items=True)
    als = {u: [item_ids[j] for j in row] for u, row in zip(users, rec_idx)}

    # 3. свежее популярное (одинаково для всех)
    recent = train[train["timestamp"] >= cutoff - GAP - DAY]
    top = recent["item_id"].value_counts().index[:k_pop].tolist()
    pop = {u: top for u in users}

    src = {"history": hist, "als": als, "popular": pop}
    union = {u: set(hist.get(u, [])) | set(als[u]) | set(pop[u]) for u in users}
    seen = train.groupby("uid")["item_id"].apply(set).to_dict()
    return src, union, truth, seen

def cand_recall(cands, truth):
    return np.mean([len(set(cands.get(u, [])) & t) / len(t) for u, t in truth.items()])

src, union, truth, seen = build_candidates(pos, val_cutoff)

for name, c in src.items():
    print(name, round(cand_recall(c, truth), 4))
print("union", round(cand_recall(union, truth), 4))
print("средний размер набора:", np.mean([len(v) for v in union.values()]))

# отдельно: повторы и новые треки в truth
truth_old = {u: t & seen[u] for u, t in truth.items() if t & seen[u]}
truth_new = {u: t - seen[u] for u, t in truth.items() if t - seen[u]}
print("union, повторы:", round(cand_recall(union, truth_old), 4))
print("union, новые:  ", round(cand_recall(union, truth_new), 4))

In [ ]:
def hist_cands(train, cutoff, tau, k):
    dt = float(cutoff - GAP) - train["timestamp"].values.astype("float64")
    w = np.exp(-dt / (tau * DAY))
    s = (train[["uid", "item_id"]].assign(w=w)
              .groupby(["uid", "item_id"], as_index=False)["w"].sum()
              .sort_values(["uid", "w"], ascending=[True, False]))
    return s.groupby("uid")["item_id"].apply(lambda x: x.tolist()[:k]).to_dict()

train, truth = split_at(pos, val_cutoff)
train = train[train["uid"].isin(truth)]          # только тестовые пользователи, быстрее
seen = train.groupby("uid")["item_id"].apply(set).to_dict()
truth_old = {u: t & seen[u] for u, t in truth.items() if t & seen[u]}

sizes = pd.Series({u: len(s) for u, s in seen.items()})
print("медиана разных треков в истории:", sizes.median(), "доля с >200:", (sizes > 200).mean())

for tau in [14, 100]:
    for k in [200, 500, 1000]:
        c = hist_cands(train, val_cutoff, tau, k)
        print(f"tau={tau} k={k}: повторы={cand_recall(c, truth_old):.4f}, "
              f"размер={np.mean([len(v) for v in c.values()]):.0f}")

c14 = hist_cands(train, val_cutoff, 14, 200)
c100 = hist_cands(train, val_cutoff, 100, 200)
both = {u: set(c14[u]) | set(c100[u]) for u in c14}
print("tau14+tau100, k=200 каждый:", round(cand_recall(both, truth_old), 4),
      np.mean([len(v) for v in both.values()]))

In [ ]:
import pickle

def build_table(pos, cutoff, k_hist=500, k_als=100, k_pop=100,
                factors=256, alpha=0.3):
    train, truth = split_at(pos, cutoff)
    train = train[["uid", "item_id", "timestamp"]].astype("int64")
    users = np.array(list(truth), dtype="int64")
    t_ref = float(cutoff - GAP)
    age_all = t_ref - train["timestamp"].values.astype("float64")

    # признаки трека (по всем пользователям)
    item_feat = pd.DataFrame({
        "pop_all": train.groupby("item_id").size(),
        "pop_7d":  train[age_all <= 7 * DAY].groupby("item_id").size(),
        "pop_1d":  train[age_all <= DAY].groupby("item_id").size(),
    }).fillna(0)

    # признаки пары (пользователь, трек) из истории
    tu = train[train["uid"].isin(users)]
    age = t_ref - tu["timestamp"].values.astype("float64")
    ui = pd.DataFrame({"uid": tu["uid"].values, "item_id": tu["item_id"].values, "age": age})
    for t in [1, 7, 14, 30, 100]:
        ui[f"w{t}"] = np.exp(-age / (t * DAY))
    ui["n7"] = (age <= 7 * DAY).astype("int32")
    g = ui.groupby(["uid", "item_id"]).agg(
        n=("age", "size"), last_days=("age", "min"), first_days=("age", "max"),
        n7=("n7", "sum"), w1=("w1", "sum"), w7=("w7", "sum"), w14=("w14", "sum"),
        w30=("w30", "sum"), w100=("w100", "sum")).reset_index()
    g["last_days"] /= DAY
    g["first_days"] /= DAY
    g["hist_rank"] = g.groupby("uid")["w14"].rank(ascending=False, method="first")

    # признаки пользователя
    user_feat = pd.DataFrame({
        "u_n": ui.groupby("uid").size(),
        "u_n7": ui.groupby("uid")["n7"].sum(),
        "u_distinct": g.groupby("uid").size(),
    })

    # кандидаты: iALS (только новое для пользователя)
    mat, user_ids, item_ids = build_matrix(train)
    mat.data = 1 + alpha * mat.data
    model = AlternatingLeastSquares(factors=factors, regularization=0.01,
                                    iterations=15, random_state=42)
    model.fit(mat)
    u_index = {u: i for i, u in enumerate(user_ids)}
    idx = np.array([u_index[u] for u in users])
    rec_idx, rec_sc = model.recommend(idx, mat[idx], N=k_als, filter_already_liked_items=True)
    item_arr = np.array(item_ids, dtype="int64")
    als = pd.DataFrame({"uid": np.repeat(users, k_als),
                        "item_id": item_arr[rec_idx].ravel(),
                        "als_rank": np.tile(np.arange(1, k_als + 1), len(users)),
                        "als_score": rec_sc.ravel()})

    # кандидаты: свежее популярное
    top = train.loc[age_all <= DAY, "item_id"].value_counts().index[:k_pop].to_numpy()
    pop = pd.DataFrame({"uid": np.repeat(users, len(top)), "item_id": np.tile(top, len(users))})

    # объединение и признаки
    cand = pd.concat([g.loc[g["hist_rank"] <= k_hist, ["uid", "item_id"]],
                      als[["uid", "item_id"]], pop]).drop_duplicates()
    cand = (cand.merge(g, on=["uid", "item_id"], how="left")
                .merge(als, on=["uid", "item_id"], how="left")
                .merge(item_feat, left_on="item_id", right_index=True, how="left")
                .merge(user_feat, left_on="uid", right_index=True, how="left"))
    for c in ["n", "n7", "w1", "w7", "w14", "w30", "w100"]:
        cand[c] = cand[c].fillna(0)
    cand["n_share"] = cand["n"] / cand["u_n"]

    # метка
    lab = pd.DataFrame([(u, i) for u, s in truth.items() for i in s],
                       columns=["uid", "item_id"]).astype("int64")
    lab["y"] = 1
    cand = cand.merge(lab, how="left", on=["uid", "item_id"])
    cand["y"] = cand["y"].fillna(0).astype("int8")

    fl = cand.select_dtypes("float64").columns
    cand[fl] = cand[fl].astype("float32")
    return cand, truth

# сначала одна отсечка, проверить
cand, truth = build_table(pos, test_start - DAY)
hits = cand.groupby("uid")["y"].sum()
n_truth = pd.Series({u: len(t) for u, t in truth.items()})
n_truth.index = n_truth.index.astype("int64")
print("shape:", cand.shape)
print("кандидатов на пользователя:", len(cand) / cand["uid"].nunique())
print("доля y=1:", cand["y"].mean())
print("candidate recall (общий):", (hits.reindex(n_truth.index).fillna(0) / n_truth).mean())

In [ ]:
import pickle

for s in [1, 2, 3, 4]:
    cand, truth = build_table(pos, test_start - s * DAY)
    cand.to_parquet(f"table_shift{s}.parquet")
    pickle.dump(truth, open(f"truth_shift{s}.pkl", "wb"))
    print("готово shift", s, cand.shape)

In [ ]:
import pickle
import pandas as pd, numpy as np
from catboost import CatBoostClassifier

tr = pd.concat([pd.read_parquet(f"table_shift{s}.parquet") for s in [2, 3, 4]], ignore_index=True)
va = pd.read_parquet("table_shift1.parquet")
truth_va = pickle.load(open("truth_shift1.pkl", "rb"))

FEATURES = [c for c in tr.columns if c not in ("uid", "item_id", "y")]
print(len(FEATURES), FEATURES)   # ожидаем 19

def rank_metrics(df, score, truth, k=100):
    d = (df[["uid", "item_id"]].assign(s=score)
           .sort_values(["uid", "s"], ascending=[True, False])
           .groupby("uid").head(k))
    recs = d.groupby("uid")["item_id"].apply(list).to_dict()
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)

# контроль: сортировка по w14 на тех же кандидатах и той же отсечке
print("baseline w14, shift 1:", rank_metrics(va, va["w14"].values, truth_va))

model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6,
                           eval_metric="Logloss", random_seed=42, verbose=50)
model.fit(tr[FEATURES], tr["y"], eval_set=(va[FEATURES], va["y"]), use_best_model=False)

p = model.predict_proba(va[FEATURES])[:, 1]
print("CatBoost, shift 1:", rank_metrics(va, p, truth_va))

imp = pd.Series(model.get_feature_importance(), index=FEATURES).sort_values(ascending=False)
print(imp)

In [ ]:
import time

groups = {
    "full":            [],
    "no_ials":         ["als_rank", "als_score"],
    "no_popularity":   ["pop_all", "pop_7d", "pop_1d"],
    "no_multi_tau":    ["w1", "w7", "w30", "w100"],
    "no_user_feats":   ["u_n", "u_n7", "u_distinct"],
    "no_hist_extra":   ["hist_rank", "n_share", "first_days", "n", "n7"],
}

def run(drop, seed=42):
    feats = [f for f in FEATURES if f not in drop]
    m = CatBoostClassifier(iterations=200, learning_rate=0.15, depth=6,
                           random_seed=seed, verbose=0)
    m.fit(tr[feats], tr["y"])
    p = m.predict_proba(va[feats])[:, 1]
    return rank_metrics(va, p, truth_va)

rows = []
experiments = [("full_seed42", [], 42), ("full_seed43", [], 43)] + \
              [(name, drop, 42) for name, drop in groups.items() if name != "full"]

for name, drop, seed in experiments:
    t0 = time.time()
    r, n = run(drop, seed)
    rows.append(dict(experiment=name, dropped=",".join(drop), recall=r, ndcg=n))
    print(rows[-1], f"{time.time()-t0:.0f}s")
    pd.DataFrame(rows).to_csv("ablation_partial.csv", index=False)

abl = pd.DataFrame(rows)
abl["d_ndcg_vs_full"] = abl["ndcg"] - abl.loc[0, "ndcg"]
abl.to_csv("ablation.csv", index=False)
print(abl)

In [ ]:
import time, pickle
import numpy as np, pandas as pd
from catboost import CatBoostRanker, Pool

t0 = time.time()

# --- данные: обучение shift 2,3,4; проверка shift 1 ---
tr = pd.concat([pd.read_parquet(f"table_shift{s}.parquet").assign(shift=s) for s in [2, 3, 4]],
               ignore_index=True)
va = pd.read_parquet("table_shift1.parquet")
truth_va = pickle.load(open("truth_shift1.pkl", "rb"))

FEATURES = [c for c in tr.columns if c not in ("uid", "item_id", "y", "shift")]
print(len(FEATURES), "признаков")

# --- уменьшаем обучающую выборку: все y=1 и 25% случайных y=0 (только в обучении) ---
rng = np.random.RandomState(42)
keep = (tr["y"].values == 1) | (rng.rand(len(tr)) < 0.25)
tr = tr[keep].copy()
print("обучающих строк после подвыборки негативов:", len(tr), "доля y=1:", tr["y"].mean())

# --- группы: пара (отсечка, пользователь) должна быть уникальной, строки группы идут подряд ---
tr["group"] = tr["shift"].astype(str) + "_" + tr["uid"].astype(str)
tr = tr.sort_values("group", kind="stable").reset_index(drop=True)
va = va.sort_values("uid", kind="stable").reset_index(drop=True)

pool_tr = Pool(tr[FEATURES], tr["y"], group_id=tr["group"].values)
pool_va = Pool(va[FEATURES], va["y"], group_id=va["uid"].values)

# --- модель (200 деревьев, lr=0.15, как в ablation, для честного сравнения) ---
rk = CatBoostRanker(loss_function="YetiRank", iterations=200, learning_rate=0.15,
                    depth=6, random_seed=42, verbose=50)
rk.fit(pool_tr)

# --- оценка на полном наборе кандидатов shift 1 ---
scores = rk.predict(pool_va)
r, n = rank_metrics(va, scores, truth_va)



print(f"YetiRank, shift 1: recall={r:.4f}, ndcg={n:.4f}")
print("Для сравнения: классификатор full_seed42 = 0.3623 / 0.2834, sort by w14 = 0.3236 / 0.2496")
print(f"время: {time.time() - t0:.0f}s")

pd.Series(rk.get_feature_importance(), index=FEATURES).sort_values(ascending=False).to_csv("importance_ranker.csv")

In [ ]:
imp = pd.Series(rk.get_feature_importance(pool_tr, type="LossFunctionChange"), index=FEATURES)
imp.sort_values(ascending=False).to_csv("importance_ranker.csv")
print(imp.sort_values(ascending=False))

In [ ]:
#Финал-1
import pickle
cand, truth = build_table(pos, test_start)
cand.to_parquet("table_shift0.parquet")
pickle.dump(truth, open("truth_shift0.pkl", "wb"))

hits = cand.groupby("uid")["y"].sum()
n_truth = pd.Series({u: len(t) for u, t in truth.items()})
n_truth.index = n_truth.index.astype("int64")
print("shape:", cand.shape, "пользователей:", cand["uid"].nunique())
print("candidate recall (потолок):", (hits.reindex(n_truth.index).fillna(0) / n_truth).mean())

In [ ]:
#Финал-2
import time
import numpy as np, pandas as pd
from catboost import CatBoostRanker, Pool

t0 = time.time()

# обучение: shift 1-4 (все метки лежат строго до начала теста)
tr = pd.concat([pd.read_parquet(f"table_shift{s}.parquet").assign(shift=s) for s in [1, 2, 3, 4]],
               ignore_index=True)
te = pd.read_parquet("table_shift0.parquet")
truth_te = pickle.load(open("truth_shift0.pkl", "rb"))

FEATURES = [c for c in tr.columns if c not in ("uid", "item_id", "y", "shift")]
assert len(FEATURES) == 19

# подвыборка негативов только в обучении (как на validation)
rng = np.random.RandomState(42)
keep = (tr["y"].values == 1) | (rng.rand(len(tr)) < 0.25)
tr = tr[keep].copy()
tr["group"] = tr["shift"].astype(str) + "_" + tr["uid"].astype(str)
tr = tr.sort_values("group", kind="stable").reset_index(drop=True)
te = te.sort_values("uid", kind="stable").reset_index(drop=True)
print("обучающих строк:", len(tr), "| тестовых строк:", len(te))

pool_tr = Pool(tr[FEATURES], tr["y"], group_id=tr["group"].values)
pool_te = Pool(te[FEATURES], te["y"], group_id=te["uid"].values)

results = {"sort_by_w14": rank_metrics(te, te["w14"].values, truth_te)}
print("sort by w14, TEST:", results["sort_by_w14"])

for seed in [42, 43, 44]:
    rk = CatBoostRanker(loss_function="YetiRank", iterations=200, learning_rate=0.15,
                        depth=6, random_seed=seed, verbose=0)
    rk.fit(pool_tr)
    results[f"yetirank_seed{seed}"] = rank_metrics(te, rk.predict(pool_te), truth_te)
    print(f"YetiRank seed {seed}, TEST:", results[f"yetirank_seed{seed}"],
          f"({time.time() - t0:.0f}s)")

yr = np.array([v for k, v in results.items() if k.startswith("yetirank")])
print("YetiRank, среднее по seed'ам:", yr.mean(axis=0), "| разброс (std):", yr.std(axis=0))

pd.DataFrame(results, index=["recall", "ndcg"]).T.to_csv("test_results.csv")
imp = pd.Series(rk.get_feature_importance(type="PredictionValuesChange"), index=FEATURES)
imp.sort_values(ascending=False).to_csv("importance_ranker.csv")
print(imp.sort_values(ascending=False))

In [ ]:
print("Popular 1 день:", eval_popular(pos, test_start, 1))
print("Мои топ-треки (счётчик):", eval_my_top(pos, test_start))
print("Мои топ-треки, затухание τ=14:", eval_my_top_decay(pos, test_start, 14))
print("iALS (постановка A, дефолт) ")